# Simple MusDis Inference

Load audio → Encode to latent → Extract structure/timbre → Reconstruct → Decode to audio

In [1]:
# Setup
import torch
import gin
import sys
from pathlib import Path
import os
import numpy as np
from IPython.display import display, Audio

# Add src to path
sys.path.append(str(Path.cwd().parent.parent / "src"))

import musdis
from music2latent import EncoderDecoder

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize music2latent encoder/decoder
encdec = EncoderDecoder()
print("Music2latent encoder/decoder loaded")

/home/lauraibnz/Code/MusDis/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
Music2latent encoder/decoder loaded


In [ ]:
# Load model
model_path = "../runs/test_ema_cosine_fix"
config_path = os.path.join(model_path, "config.gin")
checkpoint_path = os.path.join(model_path, "best_model.pt")

# Parse config and create model using gin
gin.clear_config()
gin.parse_config_file(config_path)

# Import after gin config is loaded to ensure proper registration
from musdis.pipeline.model import Diffusion
model = Diffusion()  # This will automatically use all gin configurations
model = model.to(device)

# Load checkpoint
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'], strict=False)
model.eval()
print(f"Model loaded from epoch {checkpoint['epoch']}")

Model loaded from epoch 32


In [3]:
# Step 1: Load audio and latent from dataset
# Load dataset and get a sample using the SAME split as training
from musdis.dataset.dataset import BaseDataset
from musdis.pipeline.utils import create_dataloaders
import torch
import random

# Set the same random seed as training to ensure consistent splits
torch.manual_seed(42)  # Same seed as in train.py

dataset = BaseDataset("../dataset/babyslakh_16k/", return_latents=True)
print(f"Dataset size: {len(dataset)}")

# Use the same DataLoader creation as training (now fixed)
# This ensures we get the exact same train/val split as during training
train_loader, val_loader = create_dataloaders(
    dataset, 
    batch_size=1,      # Small batch for inference
    train_split=0.95,  # Same as default in utils
    num_workers=0      # No multiprocessing for simplicity
)

print(f"Train size: {len(train_loader.dataset)}, Val size: {len(val_loader.dataset)}")

# Choose your sample here - change this number to get different samples!
sample_idx = 4  # Change this to 0, 1, 2, 3, etc. for different samples

# Or use random sampling:
# sample_idx = random.randint(0, len(val_loader.dataset) - 1)

print(f"Using sample index: {sample_idx}")
sample = val_loader.dataset[sample_idx]

# Handle different tensor shapes robustly
if sample['audio'].dim() == 1:
    sample_audio = sample['audio'].numpy()
else:
    sample_audio = sample['audio'][0].numpy() if sample['audio'].shape[0] == 1 else sample['audio'].numpy()

if sample['latent'].dim() == 2:
    latent = sample['latent'].unsqueeze(0).to(device)
else:
    latent = sample['latent'][0].unsqueeze(0).to(device) if sample['latent'].shape[0] == 1 else sample['latent'].unsqueeze(0).to(device)

print(f"Audio shape: {sample_audio.shape}")
print(f"Pre-computed latent shape: {latent.shape}")

# Play original audio from validation set (16kHz sample rate)
print("Original audio from validation set (unseen during training):")
display(Audio(sample_audio, rate=16000))

Dataset size: 2832
Train size: 2690, Val size: 142
Using sample index: 4
Audio shape: (160000,)
Pre-computed latent shape: torch.Size([1, 64, 38])
Original audio from validation set (unseen during training):


In [4]:
# Step 2: Extract structure and timbre embeddings
# Ensure latent is in the correct dtype for the model
latent = latent.to(device=device, dtype=torch.float32)
print(f"Latent dtype: {latent.dtype}, device: {latent.device}")

with torch.no_grad():
    structure_embedding, timbre_embedding = model.encode_conditioning(latent, latent)

if structure_embedding is not None:
    print(f"Structure embedding shape: {structure_embedding.shape}")  # Should preserve time
if timbre_embedding is not None:
    print(f"Timbre embedding shape: {timbre_embedding.shape}")        # Should be global

Latent dtype: torch.float32, device: cuda:0
Structure embedding shape: torch.Size([1, 64, 38])


In [8]:
# Step 3: Reconstruct latent using diffusion sampling
print("Reconstructing latent using diffusion sampling...")

with torch.no_grad():
    reconstructed_latent = model.sample(
        time_source=latent,
        cond_source=latent, 
        num_inference_steps=50  # Use fewer steps for faster inference
    )

print(f"Reconstructed latent shape: {reconstructed_latent.shape}")
print(f"Original std: {latent.std().item():.4f}, Reconstructed std: {reconstructed_latent.std().item():.4f}")
print(f"Reconstructed range: [{reconstructed_latent.min().item():.4f}, {reconstructed_latent.max().item():.4f}]")

Reconstructing latent using diffusion sampling...
Reconstructed latent shape: torch.Size([1, 64, 38])
Original std: 1.3286, Reconstructed std: 7297.9116
Reconstructed range: [-34196.2500, 35634.3047]


In [9]:
# Step 4: Decode latent back to audio
print("Decoding reconstructed latent back to audio...")

# Convert back to numpy for music2latent decoder
reconstructed_np = reconstructed_latent.cpu().numpy()
original_latent_np = latent.cpu().numpy()

# Decode both to audio
reconstructed_audio = encdec.decode(reconstructed_np)
original_latent_audio = encdec.decode(original_latent_np)

print(f"Reconstructed audio shape: {reconstructed_audio.shape}")
print(f"Original latent audio shape: {original_latent_audio.shape}")

# Play all audio for comparison (16kHz sample rate)
print("Audio comparison:")
print("1. Original audio from dataset:")
display(Audio(sample_audio, rate=16000))

print("2. Original pre-computed latent → decoded:")
display(Audio(original_latent_audio, rate=16000))

print("3. Diffusion reconstructed latent → decoded:")
display(Audio(reconstructed_audio, rate=16000))

Decoding reconstructed latent back to audio...
Reconstructed audio shape: torch.Size([1, 157184])
Original latent audio shape: torch.Size([1, 157184])
Audio comparison:
1. Original audio from dataset:


2. Original pre-computed latent → decoded:


3. Diffusion reconstructed latent → decoded:


In [10]:
# Step 5: Quality check - latent space reconstruction
import torch.nn.functional as F

with torch.no_grad():
    mse_loss = F.mse_loss(reconstructed_latent, latent)
    print(f"Latent reconstruction MSE: {mse_loss.item():.6f}")
    print("(Lower MSE indicates better reconstruction quality)")

Latent reconstruction MSE: 53268380.000000
(Lower MSE indicates better reconstruction quality)
